# Imports

In [ ]:
# my functions
import helper_functions as hf

# data handling 
import pandas as pd 
import geopandas as gpd
import numpy as np

# plotting 
import matplotlib.pyplot as plt
from matplotlib.figure import Figure
import seaborn as sns 

# mesa stuff
import mesa
from mesa import Agent, Model
from mesa.space import ContinuousSpace
from mesa.visualization import SolaraViz, make_plot_component, make_space_component
from mesa.visualization.utils import update_counter
from mesa.datacollection import DataCollector
from mesa.batchrunner import batch_run
#import solara

# rando stuff
import time
from scipy.stats import truncnorm
from scipy.stats import skewnorm
import random

# set notebook options
pd.set_option('display.max_rows', 1000)


# Load Road and Modify

In [ ]:
#road_gdf = gpd.read_file("data/roads/hw210_w_speed_limits.geojson")
road_gdf = gpd.read_parquet("data/roads/hw210_w_speed_limits.parquet")
road_gdf = road_gdf.to_crs(epsg=32612)
print(f'number of road points: {len(road_gdf)}')
road_gdf.plot()
print('')
#print('GDF Head', f'GDF CRS:{road_gdf.crs} ')
display(road_gdf.head(3))

road_gdf.crs 


# Adjust Speed Helper Functions

In [ ]:
def get_deceleration(how='soft'):
    options = {'soft': hf.get_mps(1), #~1mph/s 
               'normal': hf.get_mps(2.5), #~2.5mph/s
               'hard': hf.get_mps(5) #~5mph/s
              }
    return  options[how]
    
def build_empirical_accel_function(pctile):
    """
    Builds a function that estimates acceleration (in m/s²)
    given speed (in mph), using empirical acceleration data
    from real-world stop sign behavior.

    Returns:
        accel(speed_mph): callable function
    """
    trimmed_pctile = np.clip(pctile, .003, .97)
    
    # differnet dists in m/s^s 
    dist0 = skewnorm(loc=1, scale=0.35, a=1)
    dist1 = skewnorm(loc=2.5, scale=0.4, a=1)
    dist2 = skewnorm(loc=2, scale=0.4, a=1)
    dist3 = skewnorm(loc=1.5, scale=0.3, a=1)
    
    # Acceleration values in G, time intervals in seconds
    segments = [
        {"start_t": 0, "end_t": 1, "accel_mpss": dist0.ppf(trimmed_pctile)},
        {"start_t": 1, "end_t": 3, "accel_mpss": dist1.ppf(trimmed_pctile)},
        {"start_t": 3, "end_t": 5, "accel_mpss": dist2.ppf(trimmed_pctile)},
        {"start_t": 5, "end_t": 7, "accel_mpss": dist3.ppf(trimmed_pctile)},
    ]

    # Convert to speed ranges in mph
    speed_bounds = [0]
    for seg in segments:
        delta_v_mps = seg["accel_mpss"] * (seg["end_t"] - seg["start_t"])
        delta_v_mph = delta_v_mps * 2.237  # convert m/s to mph
        speed_bounds.append(speed_bounds[-1] + delta_v_mph)

    # Pre-compute acceleration in m/s² for each segment
    accels_mps2 = [seg["accel_mpss"] for seg in segments]
    def accel(speed_mph):
        for i in range(len(speed_bounds) - 1):
            if speed_bounds[i] <= speed_mph < speed_bounds[i + 1]:
                return accels_mps2[i]
        return accels_mps2[-1] # outputs accel in mps2 

    return accel 

def vert_decel(slope_deg):
    slope_rad = np.radians(slope_deg)
    vert_decel = 9.81* np.sin(slope_rad)
    return vert_decel

# RoadSegmentAgent

In [ ]:
class RoadSegmentAgent(mesa.Agent):
    """Represents a segment of the road. Only one car can occupy it at a time."""
    
    def __init__(self, model, position, speed_limit, linked_coord):
        super().__init__(model)
        self.position = position  # The index of the segment
        self.occupied = False  # Whether a car is on this segment
        self.status = 'im just a road'
        self.speed_limit = speed_limit
        self.linked_coord = linked_coord

    def step(self):
        """Tracks occupancy but does not move."""
        pass

# VehicleAgent base class

In [ ]:
class VehicleAgent(mesa.Agent):
    def __init__(self, model):
        super().__init__(model)
        self.status = "driving"
        self.speed = hf.get_mps(1) # starting speed
        self.speed_change = 0
        
        # For data collection 
        self.created_at_step = self.model.steps 
        self.steps_taken = 0 
        self.distance_traveled = 0
        self.car_interactions = 0
        self.gap = 0 
        self.next_agent = None
        self.driving_action = None

        # These will be set in subclasses
        self.max_speed = None
        self.ideal_distance_multiplier = None
        self.acceptable_over = None

        # Speed control tuning parameters (can be overridden)
        self.accel_curve = None
        self.jitter_threshold = hf.get_mps(2)
        self.jitter_variance = hf.get_mps(0.2)
        self.jitter = 0 
        self.max_acceleration = hf.get_mps(4)
        self.min_speed = hf.get_mps(1)
        self.deceleration_mode = 'normal'

        # init all the road segement data 
        self.road_segments = self.model.agents.select(agent_type=RoadSegmentAgent)

        # Establish the Vehicles positio n 
        self.path = self.road_segments.get('position')
        self.path_index = 0
        self.model.space.place_agent(self, self.path[0])  # <-- here is the intial place agent

        
    def end_of_road(self):
        '''If at the last segment, remove the vehicle'''
        if self.path_index >= len(self.path) - 1:
            self.status = "arrived"
            
            self.model.finished_agents.append({
                "AgentID": self.unique_id,
                'AgentType': self.__class__.__name__,
                "created_at_step": self.created_at_step,
                "steps_taken": self.steps_taken,
                "car_interactions": self.car_interactions, 
                "distance_traveled": hf.meters_to_miles(self.distance_traveled ), 
                "approx_average_mph": hf.meters_to_miles(self.distance_traveled)/(self.steps_taken/3600), 
                "acceptable_over": hf.get_mph(self.acceptable_over),
                "ideal_distance_multiplier":self.ideal_distance_multiplier
            # Add more if needed
            })
            self.remove() 
            return True


    def get_next_agent(self): 
        '''
        Used in the get_gap function
        Takes self, checks if a next_agent exists and status == driving, if so uses that, if not trys to find a new next agent. 
        '''
        # check if 1) next car is already saved & 2)it is driving. This works because self.next_agent existing is tested first
        if self.next_agent and self.next_agent.status == "driving":
             return
        
        # if the next agent does not exist look for a new next_agent
        other_vehicles = self.model.agents.select(agent_type=VehicleAgent)
        cars_ahead = [
            agent for agent in other_vehicles
            if agent.distance_traveled > self.distance_traveled
        ]
        
        # set the next agent to the next vehicle, if no next vehicle then set to None
        if cars_ahead:
            self.next_agent = min(cars_ahead, key=lambda agent: agent.distance_traveled)
        else:
            self.next_agent = None
            
        
    def get_gap(self):
            """
            Returns:
                ideal_gap: float — the desired following distance (deg)
                gap: float — distance to the closest vehicle ahead (deg)

            Used in the adjust_speed function
            """
            ideal_gap = self.speed * self.ideal_distance_multiplier  
            # run the get_next_agent function 
            self.get_next_agent()
            
            # if a agent exists then measure the gap or set to inf 
            if self.next_agent: 
                gap = model.space.get_distance(self.pos, self.next_agent.pos)
            else:
                gap = float("inf")

            self.gap = gap
            return ideal_gap, gap
        
    def get_speed_limit(self):
        return hf.get_mps(self.road_segments[self.path_index].speed_limit) + self.acceptable_over

    def smooth_brake(self, gap, ideal_gap):
        """
        Calculate deceleration based on a smooth braking rule,
        where speed and distance are in degrees per second and degrees.
        """
        if ideal_gap <= 0 or np.isnan(ideal_gap):
            force = 0
        else:
            force = max((ideal_gap - gap)/ideal_gap, 0) # this produces a number between 0 and 1
    def less_smooth_brake(self, gap, ideal_gap):
        """
        Simulate more realistic, human-like braking behavior.
        Returns a value between 0 and 1 indicating brake intensity.
        """
        if ideal_gap <= 0 or np.isnan(ideal_gap):
            return 0
        force = max((ideal_gap - gap) / ideal_gap, 0)
        # Squared to overreact when too close
        base = force ** 2
        # Add slight panic behavior for very short gaps
        if gap < 0.5 * ideal_gap:
            base += 0.2
        # Add some human-like noise
        noise = np.random.normal(0, 0.03)
        break_pct =  np.clip(base + noise, 0, 1)

        deceleration = break_pct * hf.get_mps(5) # <- this is acting as max decel 
        return deceleration
            
    def adjust_speed(self):
        ''' takes self from self uses'''
        
        ideal_gap, gap  = self.get_gap() 
        speed_limit = self.get_speed_limit()

        # save the current speed 
        old_speed = self.speed 
        
        # 1) measues the gap to the next vehicle, if less than the ideal gap, applies the smooth breaking
        if gap < ideal_gap:
            self.driving_action = 'smooth_break'
            self.car_interactions += 1
            self.speed -= self.less_smooth_brake(gap=gap, ideal_gap=ideal_gap)
            
        # 2)if the gap is > than the ideal gap see if the car vehicle is around the speed limit, if so adjust by the jitter
        elif abs(self.speed - speed_limit) < self.jitter_threshold:
            if self.driving_action != 'jitter': # ie the last action was some other set a new jitter
                self.jitter = self.random.choice([-1, 1]) * skewnorm.rvs(a=8, loc=self.jitter_variance/1.5, scale=self.jitter_variance)
            self.speed += self.jitter
            self.driving_action = 'jitter'
            
        # 3) if outside the jitter threashhold see if the car is above speed limit, if so break
        elif self.speed > speed_limit:
            self.driving_action = 'speed_limit_break'
            self.speed = max(self.speed - get_deceleration(how=self.deceleration_mode), self.min_speed)
            
        # 4) if outside the jitter threashhold & below speed limit & max speed then speed up 
        elif self.speed < self.max_speed:
            self.driving_action = 'accelerate'
            # self.speed += min(get_acceleration(self.speed, self.max_speed, self.max_acceleration), speed_limit - self.speed)
            self.speed+= self.accel_curve(hf.get_mph(self.speed))
        
        # overwrites
        if self.speed > gap: 
            self.driving_action = 'prevent_pass'
            self.speed = gap-1 
        
        # new speed - old speed
        self.speed_change = self.speed - old_speed
        

    def move_along_path(self):
        """Move the agent along its predefined path based on current speed."""
        distance_to_travel = self.speed
        self.distance_traveled += distance_to_travel
        pos = np.array(self.pos)
        new_position = pos
        
        while distance_to_travel > 0 and not self.end_of_road():
            next_target = np.array(self.path[self.path_index + 1])
            direction = model.space.get_heading(pos, next_target)
            distance = model.space.get_distance(pos, next_target)
    
            if distance < distance_to_travel:
                self.path_index += 1
                distance_to_travel -= distance
                pos = next_target
                new_position = pos
            else:
                step_vector = distance_to_travel * direction / distance
                new_position = pos + step_vector
                distance_to_travel = 0
    
        self.model.space.move_agent(self, tuple(new_position))

    def step(self):
        
        if self.status == "arrived":
            return 
        
        self.steps_taken += 1  
        self.adjust_speed()
        self.move_along_path()


# Spicific VehicleAgent Classes

In [ ]:
class CarAgent(VehicleAgent):
    """Represents a car moving in the canyon."""
    def __init__(self, model, road_points_gdf):
        super().__init__(model)
        self.status = "driving"  # the initial status of the car
        
        # speed perams
        self.max_speed = hf.get_mps(80)
        self.acceptable_over = hf.get_mps(truncnorm((-2 - 3)/4, (20 - 3)/4, loc=3, scale=4).rvs()) # this is a right skewed normal dist bounded by (-2,20)
        self.ideal_distance_multiplier = truncnorm((1.2 - 1.5)/.2, (2.5 - 1.5)/.2, loc=1.5, scale=.2).rvs()
        self.accel_curve = build_empirical_accel_function(np.random.rand())
        
        # Speed control tuning parameters 
        self.jitter_threshold = hf.get_mps(2)
        self.jitter_variance = hf.get_mps(0.2)
        self.max_acceleration = hf.get_mps(4)
        self.min_speed = hf.get_mps(1)
        self.deceleration_mode = 'normal'
        
class BusAgent(VehicleAgent):
    """Represents a bus moving in the canyon."""
    def __init__(self, model, road_points_gdf):
        super().__init__(model)
        self.status = "driving"  # the initial status of the car
        
        # speed perams
        self.max_speed = hf.get_mps(60)
        self.acceptable_over = 0
        self.ideal_distance_multiplier = 2.5
        self.accel_curve = build_empirical_accel_function(.1)

        
        # Speed control tuning parameters 
        self.jitter_threshold = hf.get_mps(2)
        self.jitter_variance = hf.get_mps(0.1)
        self.max_acceleration = hf.get_mps(1.5)
        self.min_speed = hf.get_mps(1)
        self.deceleration_mode = 'normal'

    

# Model 

In [ ]:

class TrafficModel(mesa.Model):
    """Mesa model simulating traffic on the canyon road with a car cap."""

    def __init__(self, road_points_gdf=None, max_steps=50000, seed=123, log_agents=False, p_generate=.001, max_cars=50, bus_interval=30, max_buses=5):
        super().__init__(seed=seed)
        #model perams
        self.road_points_gdf = road_points_gdf
        self.log_agents = log_agents
        self.max_steps = max_steps
    
        # car perams
        self.p_generate = p_generate  # Probability of new car each step
        self.max_cars = max_cars  # Maximum number of cars allowed
        
        # bus perams
        self.bus_interval = bus_interval
        self.max_buses = max_buses
        self.bus_first_departure = self.random.randint(0, 15 * 60)  # Random step between 0 and 15 mins
        self.bus_generation_started = False
        
        # verious trackers
        self.too_close_tracker = 0 
        self.cars_generated = 0 
        self.finished_agents = [] 
            
        # Set up ContinuousSpace
        buffer = .0001
        minx, miny, maxx, maxy = road_points_gdf.total_bounds
        self.space = ContinuousSpace(
            x_min=minx - buffer,
            x_max=maxx + buffer,
            y_min=miny - buffer,
            y_max=maxy + buffer,
            torus=False
        )

        # Create road segment agents - this just creates them in a loop setting the position via the gdf point
        self.road_segments = RoadSegmentAgent.create_agents( 
            model=self, 
            n=len(self.road_points_gdf), 
            position=[(point.x, point.y) for point in self.road_points_gdf.geometry], # need to be passed as a list
            speed_limit=[speed_limit for speed_limit in self.road_points_gdf.speed_limit],
            linked_coord=[linked_coord for linked_coord in self.road_points_gdf.linked_coord]
        )
        # place all the road segments in space - goes hand in hand with read point reation 
        for agent, point in zip(self.road_segments, road_points_gdf.geometry):self.space.place_agent(agent, (point.x, point.y))

        # establish the data collector 
        agent_reporters={
            "AgentType": lambda a: a.__class__.__name__ ,
            'status': lambda a: a.status if isinstance(a, VehicleAgent) else None,
            'driving_action': lambda a: a.driving_action if isinstance(a, VehicleAgent) else None,
            'speed_change': lambda a: hf.get_mph(a.speed_change) if isinstance(a, VehicleAgent) else None,
            'speed_change_mps2': lambda a: a.speed_change if isinstance(a, VehicleAgent) else None,
            'speed': lambda a: hf.get_mph(a.speed) if isinstance(a, VehicleAgent) else None,
            'steps_taken': lambda a: a.steps_taken if isinstance(a, VehicleAgent) else None,
            'pos':lambda a: a.pos if isinstance(a, VehicleAgent) else None,
            'gap_ft':lambda a: hf.meters_to_feet(a.gap) if isinstance(a, VehicleAgent) else None,
            "next_vehicle": lambda a: a.next_agent.unique_id if isinstance(a, VehicleAgent) and a.next_agent is not None else None,
            "next_vehicle_status": lambda a: a.next_agent.status if isinstance(a, VehicleAgent) and a.next_agent is not None else None,


            #'hrs': lambda a: a.steps_taken/3600 if isinstance(a, CarAgent) else None,
            #'distance_traveled': lambda a: a.distance_traveled if isinstance(a, CarAgent) else None,
            #'acceptable_over': lambda a: hf.get_mph(a.acceptable_over) if isinstance(a, CarAgent) else None,
            #'ideal_distance_multiplier': lambda a: a.ideal_distance_multiplier if isinstance(a, CarAgent) else None,
        }

        model_reporters={
            "cars_generated": lambda m: m.cars_generated,
            "too_close_tracker": lambda m: m.too_close_tracker, 
            "FinishedAgentsSummary": lambda m: None  # Placeholder
        }

        if log_agents:
            self.datacollector = DataCollector(
                model_reporters = model_reporters, 
                agent_reporters = agent_reporters
            )
        else: 
            self.datacollector = DataCollector(model_reporters = model_reporters)
    
    def generate_new_car(self):
        # Only generate if under max limit
        if self.cars_generated >= self.max_cars:
            return
        # Get the starting point
        start_point = self.road_points_gdf.iloc[0].geometry.coords[0]  
        
        # Check if another car is too close to the start
        too_close = any(
            self.space.get_distance(agent.pos, start_point) < 5 # m -> degrees
            for agent in self.agents.select(agent_type=VehicleAgent)[-5:] # last 5 cars
        )
        if too_close:
            self.too_close_tracker += 1
        elif self.random.random() < self.p_generate:
            CarAgent.create_agents(model=self, n=1, road_points_gdf=self.road_points_gdf)
            self.cars_generated += 1
    
    def generate_new_bus(self):
        """
        Generate a new bus:
        - First bus is generated at a random step (0–15 mins).
        - Then follow a fixed interval based on bus_interval (in minutes).
        - Never exceed max_buses on the road.
        """
        current_step = self.steps
        steps_per_interval = self.bus_interval * 60
    
        # First departure check
        if not self.bus_generation_started:
            if current_step >= self.bus_first_departure:
                self.bus_generation_started = True
            else:
                return  # Still waiting for the randomized first departure
    
        # After the first departure
        if (current_step - self.bus_first_departure) % steps_per_interval == 0:
            active_buses = self.agents.select(agent_type=BusAgent)
            if len(active_buses) < self.max_buses:
                print('bus generated')
                BusAgent.create_agents(model=self, n=1, road_points_gdf=self.road_points_gdf)
    
    def model_stop_process(self):
        # add agent summary data to the datacollector
        self.datacollector.model_vars["FinishedAgentsSummary"][-1] = self.finished_agents
        self.running = False
        
    def step(self):

        # generate bus 
        self.generate_new_bus()
        
        # generate a new car based on a simple probability 
        self.generate_new_car()

        # Shuffle agent execution and step them - this calls the step functions of the agents
        #self.agents.shuffle_do("step")
        self.agents.do("step")
        
        # Collect data before stepping
        self.datacollector.collect(self)

        # Stop model when all generated cars have been removed
        if self.cars_generated == self.max_cars:
            remaining_cars = self.agents.select(agent_type=CarAgent)
            if len(remaining_cars) == 0:
                print("All cars have been removed. Stopping model.")
                self.model_stop_process()
        
        # Stop model at hard cap of steps
        if self.steps >= self.max_steps:
            print(f"Reached max step count ({self.max_steps}). Stopping model.")
            self.model_stop_process()
             


# Simple model run (fast)

In [ ]:
%%time
model = TrafficModel(road_points_gdf=road_gdf, max_steps=100000, log_agents=True, seed=123, p_generate=0.1, max_cars=500, bus_interval=15, max_buses=5 )

while model.running: 
    model.step()

#for i in range(15):
#    model.step()

print(f'Model ran for {model.steps} steps')

## Analyze data

### Finished agents

In [ ]:
# process the finished_agents data 
finished_agents = hf.make_finished_agents_df(model.finished_agents)
print(f'N cars: {len(finished_agents)}, fake hours: {round(model.steps/3600,1)}')

# Make the nice histogram
hf.make_travel_time_hist(finished_agents)

In [ ]:
# collect the agent data
agent_data = model.datacollector.get_agent_vars_dataframe().reset_index()
# filter for cars and buses agents and manipulate a bit
vehicles_full = agent_data.loc[agent_data.AgentType.isin(['CarAgent', 'BusAgent'])].copy()


# produce some lists of ids i might want to look at
bus_ids = list(vehicles_full.loc[vehicles_full.AgentType == 'BusAgent'].AgentID.unique())
slowest_ids = list(vehicles_full.groupby(by='AgentID', as_index=False).max('steps_taken').sort_values('steps_taken', ascending=False).AgentID[:5])

print(f'''Bus IDs: {bus_ids}
Slow IDs: {slowest_ids}''')

hf.make_driving_actions_plots(vehicles_full)

In [ ]:
# run the animation
# looking at one car
issue_car_id = None #430 

hf.animate_traffic(vehicles_full, road_gdf, interval=60, step_skip=20, watch=issue_car_id, zoom=40 )


In [ ]:
issue_step = 0

### Issue car analysis

In [ ]:
issue_car_df = vehicles_full.loc[(vehicles_full.AgentID==issue_car_id)]
print(f'Start of issue: {issue_car_df.loc[issue_car_df.speed<10]["Step"].min()}')
display(sns.scatterplot(data=issue_car_df, x='Step', y='speed', hue='driving_action', alpha=0.5))

issue_car_df.loc[issue_car_df.Step > issue_step].head(50)

In [ ]:
# looking at a group of issue cars
issue_car_ids = [issue_car_id, issue_car_id+1, issue_car_id+2]
issue_car_df = vehicles_full.loc[vehicles_full.AgentID.isin(issue_car_ids)]
print(issue_car_ids)
issue_car_df.loc[issue_car_df.Step > issue_step].head(10)

In [ ]:
def plot_agent_trajectories(df, y_var,step_range=(None, None)):
    """
    Plot agent trajectories over time using seaborn lineplot.

    Parameters:
    - df: pd.DataFrame with columns ['Step', 'AgentID', y_var]
    - y_var: str, the column to use on the y-axis

    Returns:
    - Displays a line plot where each line is one AgentID

    """
    start, end = step_range

    if start is not None:
        df = df[df['Step'] >= start]
    if end is not None:
        df = df[df['Step'] <= end]
    
    plt.figure(figsize=(10, 5))
    sns.lineplot(data=df, x="Step", y=y_var, hue="AgentID", legend=False)

     # Add labels to the start of each line
    for agent_id in df['AgentID'].unique():
        agent_df = df[df['AgentID'] == agent_id]
        first_point = agent_df.iloc[0]
        label = f"Agent:{agent_id} - ({first_point['AgentType']})"
        plt.text(first_point["Step"], first_point[y_var]+.5, label, fontsize=8, ha='left', va='top')

    plt.title(f"Agent Trajectories of {y_var} over Time", fontsize=14)
    plt.xlabel("Step")
    plt.ylabel(y_var)
    plt.tight_layout()
    plt.show()

step_range=(issue_step,issue_step+100 )
plot_agent_trajectories(issue_car_df, 'speed', step_range)

In [ ]:
# car gap over time
gap_over_time = cars_full.groupby(by='Step', as_index=False).agg({'gap_ft':'mean'})
sns.scatterplot(data=gap_over_time, x='Step', y='gap_ft')